# How network structure changes shock loss

## Four Generated Network Structures

The generated experiment uses twelve institutions. Every row of the relative-liability matrix has internal share $0.8$:

$$
\sum_{j=1}^{12}\Pi_{ij}=0.8,
$$

so the remaining $0.2$ of each payment goes outside the network. Ring, complete, core-periphery, and two-community structures alter only the allocation of the internal share. Nominal liabilities and external assets are generated visibly from fixed seeds. They are stylized values, not estimates for actual institutions.

In [1]:
from pathlib import Path
import sys

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
from IPython.display import HTML, display as _display


def display(value):
    module = value.__class__.__module__
    if module.startswith("pandas") and hasattr(value, "_repr_html_"):
        html = value._repr_html_()
        return _display(HTML('<div style="display:flex;justify-content:center;">' + html + "</div>"))
    return _display(value)
from scipy.optimize import linprog


BOOK_ROOT = Path.cwd().parents[1]

import RiskLabAI
from RiskLabAI.optimization import eisenberg_noe_clearing

assert RiskLabAI.__version__ == "3.2.0"

FIGURES = BOOK_ROOT / "outputs" / "figures" / "28_network_clearing"
TABLES = BOOK_ROOT / "outputs" / "tables" / "28_network_clearing"
FIGURE_DATA = BOOK_ROOT / "outputs" / "figure_data" / "28_network_clearing"
for directory in [FIGURES, TABLES, FIGURE_DATA]:
    directory.mkdir(parents=True, exist_ok=True)

PALETTE = {
    "background": "#FFFFFF",
    "axes": "#FFFFFF",
    "text": "#1B365D",
    "cyan": "#0072B2",
    "amber": "#C76D00",
    "green": "#00866A",
    "rose": "#B83B3B",
    "violet": "#A03A78",
    "grid": "#D9DEE5",
}
plt.rcParams.update({
    "font.family": "DejaVu Serif",
    "font.size": 11.5,
    "figure.facecolor": PALETTE["background"],
    "axes.facecolor": PALETTE["axes"],
    "axes.edgecolor": PALETTE["text"],
    "axes.labelcolor": PALETTE["text"],
    "axes.titlecolor": PALETTE["text"],
    "xtick.color": PALETTE["text"],
    "ytick.color": PALETTE["text"],
    "text.color": PALETTE["text"],
    "grid.color": PALETTE["grid"],
    "grid.linestyle": ":",
    "legend.facecolor": PALETTE["axes"],
    "legend.edgecolor": PALETTE["grid"],
    "savefig.facecolor": PALETTE["background"],
    "savefig.transparent": False,
    "savefig.dpi": 300,
})


def rendered_value(value):
    if pd.isna(value):
        return ""
    if isinstance(value, (bool, np.bool_)):
        return "Pass" if bool(value) else "Fail"
    if isinstance(value, (int, np.integer)):
        return f"{int(value):,}"
    if isinstance(value, (float, np.floating)):
        value = float(value)
        if value == 0:
            return "0"
        if value.is_integer():
            return f"{int(value):,}"
        if abs(value) < 0.0001 or abs(value) >= 1000000:
            return f"{value:.6g}"
        return f"{value:.6f}".rstrip("0").rstrip(".")
    return str(value)


def rendered_frame(frame):
    return frame.apply(lambda column: column.map(rendered_value))


def save_table(frame, stem, caption, label, fit=False):
    shown = rendered_frame(frame)
    shown.to_csv(TABLES / f"{stem}.csv", index=False)
    latex = shown.to_latex(index=False, escape=True, caption=caption, label=label, position="htbp")
    if fit:
        latex = latex.replace("\\begin{tabular}", "\\begin{adjustbox}{max width=\\textwidth}\\begin{tabular}", 1).replace("\\end{tabular}", "\\end{tabular}\\end{adjustbox}", 1)
    (TABLES / f"{stem}.tex").write_text(latex, encoding="utf-8")
    display(shown)
    return shown


def finish_axis(axis, grid=True):
    axis.spines["top"].set_visible(False)
    axis.spines["right"].set_visible(False)
    if grid:
        axis.grid(True, alpha=0.65)
    axis.set_axisbelow(True)


def save_figure(figure, stem):
    figure.savefig(FIGURES / f"{stem}.png", dpi=300, bbox_inches="tight", facecolor=PALETTE["background"], transparent=False)
    figure.savefig(FIGURES / f"{stem}.pdf", bbox_inches="tight", facecolor=PALETTE["background"], transparent=False)
    display(figure)
    plt.close(figure)


def ring_matrix(size, internal_share=0.8):
    matrix = np.zeros((size, size))
    for debtor in range(size):
        matrix[debtor, (debtor + 1) % size] = internal_share
    return matrix


def complete_matrix(size, internal_share=0.8):
    matrix = np.full((size, size), internal_share / (size - 1))
    np.fill_diagonal(matrix, 0)
    return matrix


def core_periphery_matrix(size, internal_share=0.8, core_size=4):
    matrix = np.zeros((size, size))
    core = np.arange(core_size)
    periphery = np.arange(core_size, size)
    for debtor in range(size):
        if debtor < core_size:
            other_core = core[core != debtor]
            matrix[debtor, other_core] = 0.4 / len(other_core)
            matrix[debtor, periphery] = 0.4 / len(periphery)
        else:
            matrix[debtor, core] = 0.64 / len(core)
            other_periphery = periphery[periphery != debtor]
            matrix[debtor, other_periphery] = 0.16 / len(other_periphery)
    return matrix * (internal_share / 0.8)


def community_matrix(size, internal_share=0.8):
    matrix = np.zeros((size, size))
    half = size // 2
    for debtor in range(size):
        own = np.arange(0, half) if debtor < half else np.arange(half, size)
        other = np.arange(half, size) if debtor < half else np.arange(0, half)
        own = own[own != debtor]
        matrix[debtor, own] = 0.68 / len(own)
        matrix[debtor, other] = 0.12 / len(other)
    return matrix * (internal_share / 0.8)


def greatest_iteration(external_assets, nominal_liabilities, relative_liabilities, tolerance=1e-12, maximum_steps=10000):
    payments = np.asarray(nominal_liabilities, dtype=float).copy()
    history = [payments.copy()]
    for _ in range(maximum_steps):
        updated = np.minimum(nominal_liabilities, external_assets + relative_liabilities.T @ payments)
        history.append(updated.copy())
        if np.max(np.abs(updated - payments)) <= tolerance:
            return updated, np.asarray(history)
        payments = updated
    raise RuntimeError("Monotone iteration did not converge.")


def direct_linear_program(external_assets, nominal_liabilities, relative_liabilities):
    result = linprog(
        -np.ones(len(nominal_liabilities)),
        A_ub=np.eye(len(nominal_liabilities)) - relative_liabilities.T,
        b_ub=np.minimum(external_assets, nominal_liabilities),
        bounds=np.column_stack((np.zeros(len(nominal_liabilities)), nominal_liabilities)),
        method="highs",
    )
    if not result.success:
        raise RuntimeError(result.message)
    return np.asarray(result.x)


def clearing_metrics(external_assets, nominal_liabilities, relative_liabilities):
    payments = eisenberg_noe_clearing(external_assets, nominal_liabilities, relative_liabilities)
    fixed = np.minimum(nominal_liabilities, external_assets + relative_liabilities.T @ payments)
    unpaid = np.maximum(nominal_liabilities - payments, 0)
    return {
        "payments": payments,
        "unpaid": unpaid,
        "total payments": float(payments.sum()),
        "total unpaid": float(unpaid.sum()),
        "default count": int(np.sum(unpaid > 1e-8)),
        "payment ratio": float(payments.sum() / nominal_liabilities.sum()) if nominal_liabilities.sum() else 1.0,
        "fixed point residual": float(np.max(np.abs(payments - fixed))) if len(payments) else 0.0,
    }


def network_positions(name, size):
    if name in {"Ring", "Complete"}:
        return nx.circular_layout(range(size))
    if name == "Core-periphery":
        positions = {}
        core = nx.circular_layout(range(4), scale=0.45)
        outer = nx.circular_layout(range(4, size), scale=1.0)
        positions.update(core)
        positions.update(outer)
        return positions
    positions = {}
    left = nx.circular_layout(range(size // 2), scale=0.55, center=(-0.7, 0))
    right = nx.circular_layout(range(size // 2, size), scale=0.55, center=(0.7, 0))
    positions.update(left)
    positions.update(right)
    return positions


def plot_network(relative_liabilities, name, stem, node_values=None):
    graph = nx.DiGraph()
    size = len(relative_liabilities)
    graph.add_nodes_from(range(size))
    for debtor in range(size):
        for creditor in range(size):
            weight = relative_liabilities[debtor, creditor]
            if weight > 1e-12:
                graph.add_edge(debtor, creditor, weight=weight)
    positions = network_positions(name, size)
    values = np.ones(size) if node_values is None else np.asarray(node_values)
    colors = [PALETTE["rose"] if value > 1e-8 else PALETTE["cyan"] for value in values]
    widths = [0.8 + 5 * graph.edges[edge]["weight"] for edge in graph.edges]
    figure, axis = plt.subplots(figsize=(7.2, 6.0), constrained_layout=True)
    nx.draw_networkx_nodes(graph, positions, node_color=colors, node_size=620, edgecolors=PALETTE["text"], linewidths=0.8, ax=axis)
    nx.draw_networkx_labels(graph, positions, font_color=PALETTE["background"], font_weight="bold", ax=axis)
    nx.draw_networkx_edges(graph, positions, width=widths, alpha=0.55, edge_color=PALETTE["text"], arrows=True, arrowsize=13, connectionstyle="arc3,rad=0.06", ax=axis)
    axis.set_title(name)
    axis.set_axis_off()
    save_figure(figure, stem)
    edge_data = pd.DataFrame([{"Debtor": source, "Creditor": target, "Relative liability": graph.edges[source, target]["weight"]} for source, target in graph.edges])
    edge_data.to_csv(FIGURE_DATA / f"{stem}.csv", index=False)


def majorizes(source, target, tolerance=1e-12):
    source_sorted = np.sort(np.asarray(source))[::-1]
    target_sorted = np.sort(np.asarray(target))[::-1]
    return bool(np.all(np.cumsum(source_sorted)[:-1] >= np.cumsum(target_sorted)[:-1] - tolerance) and np.isclose(source_sorted.sum(), target_sorted.sum(), atol=tolerance))

size = 12
internal_share = 0.8
outside_share = 0.2
seed = 2902
rng = np.random.default_rng(seed)
liabilities = 100 * rng.lognormal(mean=0.0, sigma=0.45, size=size)
liabilities = liabilities / liabilities.mean()
structures = {
    "Ring": ring_matrix(size, internal_share),
    "Complete": complete_matrix(size, internal_share),
    "Core-periphery": core_periphery_matrix(size, internal_share, 4),
    "Two-community": community_matrix(size, internal_share),
}
required = np.vstack([np.maximum(liabilities - matrix.T @ liabilities, 0) for matrix in structures.values()])
external_assets = required.max(axis=0) + 0.12 * liabilities
shock_node = int(np.argmax(liabilities))

contract = pd.DataFrame([
    {"Item": "Data source", "Value": "Generated in this notebook"},
    {"Item": "Public bilateral panel found", "Value": "No"},
    {"Item": "Reason generation is necessary", "Value": "Public aggregates do not identify bilateral nominal liabilities"},
    {"Item": "Institutions", "Value": size},
    {"Item": "Internal liability share", "Value": internal_share},
    {"Item": "Outside liability share", "Value": outside_share},
    {"Item": "Random seed", "Value": seed},
    {"Item": "Download during execution", "Value": "None"},
])
save_table(contract, "generated_data_contract", "Generation contract for the stylized network experiment.", "tab:chapter29-generated-data", fit=True)

structure_rows = []
row_rows = []
balance_rows = []
for name, matrix in structures.items():
    positive = matrix[matrix > 0]
    structure_rows.append({
        "Structure": name,
        "Directed internal links": int(np.sum(matrix > 0)),
        "Mean positive share": positive.mean(),
        "Mean row squared concentration": np.mean(np.sum(matrix ** 2, axis=1)),
        "Largest row share": matrix.max(),
    })
    row_rows.append({
        "Structure": name,
        "Smallest row sum": matrix.sum(axis=1).min(),
        "Largest row sum": matrix.sum(axis=1).max(),
        "Diagonal maximum": np.diag(matrix).max(),
        "Nonnegative": bool(np.all(matrix >= 0)),
        "Row contract passes": bool(np.allclose(matrix.sum(axis=1), internal_share) and np.allclose(np.diag(matrix), 0)),
    })
    full = clearing_metrics(external_assets, liabilities, matrix)
    balance_rows.append({
        "Structure": name,
        "Total nominal liabilities": liabilities.sum(),
        "Total external assets": external_assets.sum(),
        "Baseline total unpaid": full["total unpaid"],
        "Baseline default count": full["default count"],
        "Fixed point residual": full["fixed point residual"],
    })

save_table(pd.DataFrame(structure_rows), "network_structure_summary", "Link and row-concentration summaries for the four generated structures.", "tab:chapter29-structure-summary", fit=True)
save_table(pd.DataFrame(row_rows), "network_row_checks", "Row-sum, diagonal, and nonnegativity checks.", "tab:chapter29-network-row-checks", fit=True)
save_table(pd.DataFrame(balance_rows), "network_balance_sheet_summary", "Common generated balance-sheet totals and baseline clearing results.", "tab:chapter29-balance-summary", fit=True)

for name, matrix in structures.items():
    stem = name.lower().replace("-", "_").replace(" ", "_") + "_network"
    plot_network(matrix, name, stem)

,Item,Value
0,Data source,Generated in this notebook
1,Public bilateral panel found,No
2,Reason generation is necessary,Public aggregates do not identify bilateral no...
3,Institutions,12
4,Internal liability share,0.8
5,Outside liability share,0.2
6,Random seed,"2,902"
7,Download during execution,None


,Structure,Directed internal links,Mean positive share,Mean row squared concentration,Largest row share
0,Ring,12,0.8,0.64,0.8
1,Complete,132,0.072727,0.058182,0.072727
2,Core-periphery,132,0.072727,0.095149,0.16
3,Two-community,132,0.072727,0.09488,0.136


,Structure,Smallest row sum,Largest row sum,Diagonal maximum,Nonnegative,Row contract passes
0,Ring,0.8,0.8,0,Pass,Pass
1,Complete,0.8,0.8,0,Pass,Pass
2,Core-periphery,0.8,0.8,0,Pass,Pass
3,Two-community,0.8,0.8,0,Pass,Pass


,Structure,Total nominal liabilities,Total external assets,Baseline total unpaid,Baseline default count,Fixed point residual
0,Ring,12,7.869594,0,0,1.11022e-16
1,Complete,12,7.869594,0,0,1.11022e-16
2,Core-periphery,12,7.869594,0,0,1.11022e-16
3,Two-community,12,7.869594,0,0,1.11022e-16


<Figure size 720x600 with 1 Axes>

<Figure size 720x600 with 1 Axes>

<Figure size 720x600 with 1 Axes>

<Figure size 720x600 with 1 Axes>

## A Fixed Shock Grid

For shock fraction $s\in[0,1]$, only the selected institution's outside assets change:

$$
e_i(s)=
\begin{cases}
(1-s)e_i(0), & i=i^{\star},\\
e_i(0), & i\ne i^{\star}.
\end{cases}
$$

Here, $i^{\star}$ is the institution with the largest generated nominal liability. A shock fraction of zero leaves its external assets unchanged, while a fraction of one removes them. Payments are recomputed separately for every network and shock size. Changes in defaults are finite outcomes for these generated balance sheets.

In [2]:
shock_grid = np.round(np.linspace(0, 1, 21), 2)
shock_rows = []
for name, matrix in structures.items():
    for shock_fraction in shock_grid:
        shocked_assets = external_assets.copy()
        shocked_assets[shock_node] *= 1 - shock_fraction
        result = clearing_metrics(shocked_assets, liabilities, matrix)
        shock_amount = external_assets[shock_node] * shock_fraction
        shock_rows.append({
            "Structure": name,
            "Shock fraction": shock_fraction,
            "Shock amount": shock_amount,
            "Total unpaid": result["total unpaid"],
            "Default count": result["default count"],
            "Payment ratio": result["payment ratio"],
            "Loss amplification": result["total unpaid"] / shock_amount if shock_amount > 0 else 0,
            "Fixed point residual": result["fixed point residual"],
        })
shock_results = pd.DataFrame(shock_rows)
save_table(shock_results, "shock_grid_results", "Clearing results over the complete generated shock grid.", "tab:chapter29-shock-grid", fit=True)

selected = shock_results.loc[shock_results["Shock fraction"].isin([0.25, 0.5, 0.85, 1.0])].reset_index(drop=True)
save_table(selected, "selected_shock_outcomes", "Selected small, middle, and large shock outcomes.", "tab:chapter29-selected-shocks", fit=True)

threshold_rows = []
for name in structures:
    part = shock_results.loc[shock_results["Structure"] == name]
    positive = part.loc[part["Default count"] > 0]
    changes = part.loc[part["Default count"].diff().fillna(0) != 0, ["Shock fraction", "Default count"]]
    threshold_rows.append({
        "Structure": name,
        "First shock with a default": positive["Shock fraction"].min() if len(positive) else np.nan,
        "Largest default count": part["Default count"].max(),
        "Number of default-set count changes": len(changes),
        "Unpaid liabilities at full shock": part.loc[part["Shock fraction"] == 1, "Total unpaid"].iloc[0],
    })
save_table(pd.DataFrame(threshold_rows), "default_thresholds", "First observed default thresholds and full-shock losses on the fixed grid.", "tab:chapter29-default-thresholds", fit=True)

residuals = shock_results.groupby("Structure", as_index=False).agg(
    maximum_fixed_point_residual=("Fixed point residual", "max"),
    minimum_payment_ratio=("Payment ratio", "min"),
    maximum_default_count=("Default count", "max"),
)
residuals["Residual passes"] = residuals["maximum_fixed_point_residual"] <= 1e-8
save_table(residuals, "shock_fixed_point_checks", "Fixed point residuals and range checks across the shock grid.", "tab:chapter29-shock-residuals", fit=True)

for metric, ylabel, title, stem in [
    ("Total unpaid", "Total unpaid liabilities", "Unpaid Liabilities Change with Shock Size", "shock_unpaid_curves"),
    ("Default count", "Defaulted institutions", "Default Counts Change at Finite Shock Thresholds", "shock_default_counts"),
    ("Payment ratio", "Cleared share of nominal liabilities", "Payment Ratios across the Fixed Shock Grid", "shock_payment_ratios"),
]:
    plot_data = shock_results[["Structure", "Shock fraction", metric]].copy()
    plot_data.to_csv(FIGURE_DATA / f"{stem}.csv", index=False)
    figure, axis = plt.subplots(figsize=(9.4, 5.2), constrained_layout=True)
    colors = [PALETTE["cyan"], PALETTE["amber"], PALETTE["green"], PALETTE["rose"]]
    for name, color in zip(structures, colors):
        part = plot_data.loc[plot_data["Structure"] == name]
        axis.plot(part["Shock fraction"], part[metric], marker="o", markersize=3.8, linewidth=1.9, color=color, label=name)
    axis.set_xlabel("Shock fraction")
    axis.set_ylabel(ylabel)
    axis.set_title(title)
    axis.legend()
    finish_axis(axis)
    save_figure(figure, stem)

snapshot = selected.loc[selected["Shock fraction"].isin([0.25, 0.85]), ["Structure", "Shock fraction", "Total unpaid", "Default count"]].copy()
snapshot.to_csv(FIGURE_DATA / "small_large_shock_comparison.csv", index=False)
figure, axes = plt.subplots(1, 2, figsize=(10.6, 4.8), constrained_layout=True)
for axis, shock_fraction in zip(axes, [0.25, 0.85]):
    part = snapshot.loc[snapshot["Shock fraction"] == shock_fraction]
    axis.bar(part["Structure"], part["Total unpaid"], color=[PALETTE["cyan"], PALETTE["amber"], PALETTE["green"], PALETTE["rose"]])
    axis.set_title(f"Shock fraction {shock_fraction:g}")
    axis.set_ylabel("Total unpaid liabilities")
    axis.tick_params(axis="x", rotation=18)
    finish_axis(axis)
save_figure(figure, "small_large_shock_comparison")

,Structure,Shock fraction,Shock amount,Total unpaid,Default count,Payment ratio,Loss amplification,Fixed point residual
0,Ring,0,0,0,0,1,0,1.11022e-16
1,Ring,0.05,0.10809,0,0,1,0,1.11022e-16
2,Ring,0.1,0.21618,0,0,1,0,1.11022e-16
3,Ring,0.15,0.324271,0,0,1,0,1.11022e-16
4,Ring,0.2,0.432361,0,0,1,0,1.11022e-16
...,...,...,...,...,...,...,...,...
79,Two-community,0.8,1.729444,1.114327,1,0.907139,0.644327,1.11022e-16
80,Two-community,0.85,1.837534,1.222417,1,0.898132,0.665249,1.11022e-16
81,Two-community,0.9,1.945624,1.330507,1,0.889124,0.683846,1.11022e-16
82,Two-community,0.95,2.053714,1.438598,1,0.880117,0.700486,1.11022e-16


,Structure,Shock fraction,Shock amount,Total unpaid,Default count,Payment ratio,Loss amplification,Fixed point residual
0,Ring,0.25,0.540451,0,0,1,0,1.11022e-16
1,Ring,0.5,1.080902,0.418124,1,0.965156,0.386828,2.22045e-16
2,Ring,0.85,1.837534,1.174755,1,0.902104,0.639311,1.11022e-16
3,Ring,1,2.161805,1.499026,1,0.875081,0.693414,1.11022e-16
4,Complete,0.25,0.540451,0,0,1,0,1.11022e-16
5,Complete,0.5,1.080902,0.429852,1,0.964179,0.397679,1.11022e-16
6,Complete,0.85,1.837534,1.186484,1,0.901126,0.645693,2.22045e-16
7,Complete,1,2.161805,1.510754,1,0.874104,0.698839,1.11022e-16
8,Core-periphery,0.25,0.540451,0.273824,1,0.977181,0.506659,1.11022e-16
9,Core-periphery,0.5,1.080902,0.814276,1,0.932144,0.753329,2.22045e-16


,Structure,First shock with a default,Largest default count,Number of default-set count changes,Unpaid liabilities at full shock
0,Ring,0.35,1,1,1.499026
1,Complete,0.35,1,1,1.510754
2,Core-periphery,0.15,1,1,1.895178
3,Two-community,0.3,1,1,1.546688


,Structure,maximum_fixed_point_residual,minimum_payment_ratio,maximum_default_count,Residual passes
0,Complete,2.22045e-16,0.874104,1,Pass
1,Core-periphery,4.44089e-16,0.842069,1,Pass
2,Ring,2.22045e-16,0.875081,1,Pass
3,Two-community,4.44089e-16,0.871109,1,Pass


<Figure size 940x520 with 1 Axes>

<Figure size 940x520 with 1 Axes>

<Figure size 940x520 with 1 Axes>

<Figure size 1060x480 with 2 Axes>

## Repeated Generated Balance Sheets

For generated run $b$, liabilities and asset cushions vary while every network keeps the same row-sum rule. At shock sizes $s\in\{0.25,0.85\}$, the recorded loss is

$$
U_{b,m}(s)=\sum_{i=1}^{12}\left(\overline p_{b,i}-p_{b,m,i}(s)\right),
$$

where $m$ identifies the network structure. The quantity $U_{b,m}(s)$ is total unpaid liability in one generated system. Two hundred fixed seeds show how much the finite ordering depends on the generated balance sheet. They do not form a population estimate.

In [3]:
seed_rows = []
for draw in range(200):
    draw_rng = np.random.default_rng(290200 + draw)
    draw_liabilities = draw_rng.lognormal(mean=0.0, sigma=0.55, size=size)
    draw_liabilities = 100 * draw_liabilities / draw_liabilities.mean()
    draw_required = np.vstack([np.maximum(draw_liabilities - matrix.T @ draw_liabilities, 0) for matrix in structures.values()])
    cushion = draw_rng.uniform(0.04, 0.2, size) * draw_liabilities
    draw_assets = draw_required.max(axis=0) + cushion
    draw_node = int(draw_rng.integers(0, size))
    for shock_fraction in [0.25, 0.85]:
        shocked = draw_assets.copy()
        shocked[draw_node] *= 1 - shock_fraction
        for name, matrix in structures.items():
            result = clearing_metrics(shocked, draw_liabilities, matrix)
            seed_rows.append({
                "Draw": draw,
                "Structure": name,
                "Shock fraction": shock_fraction,
                "Shock node": draw_node,
                "Total unpaid": result["total unpaid"],
                "Default count": result["default count"],
                "Fixed point residual": result["fixed point residual"],
            })
seed_results = pd.DataFrame(seed_rows)

seed_contract = pd.DataFrame([
    {"Setting": "Generated runs", "Value": 200},
    {"Setting": "First seed", "Value": 290200},
    {"Setting": "Last seed", "Value": 290399},
    {"Setting": "Institutions per run", "Value": size},
    {"Setting": "Shock fractions", "Value": "0.25 and 0.85"},
    {"Setting": "Rows per structure", "Value": 400},
])
save_table(seed_contract, "seed_experiment_contract", "Fixed settings for the repeated generated balance-sheet experiment.", "tab:chapter29-seed-contract")

seed_summary = seed_results.groupby(["Shock fraction", "Structure"])["Total unpaid"].agg(["mean", "std", "median", "min", "max"]).reset_index()
save_table(seed_summary, "seed_unpaid_summary", "Distribution summaries for total unpaid liabilities over two hundred generated runs.", "tab:chapter29-seed-summary", fit=True)

rank_rows = []
for shock_fraction in [0.25, 0.85]:
    means = seed_summary.loc[seed_summary["Shock fraction"] == shock_fraction].sort_values("mean")
    for rank, row in enumerate(means.itertuples(index=False), 1):
        rank_rows.append({"Shock fraction": shock_fraction, "Mean-loss rank": rank, "Structure": row.Structure, "Mean total unpaid": row.mean})
save_table(pd.DataFrame(rank_rows), "small_large_ranking", "Finite mean-loss orderings at the two declared shock sizes.", "tab:chapter29-small-large-ranking", fit=True)

seed_results.to_csv(FIGURE_DATA / "seed_unpaid_distributions.csv", index=False)
figure, axes = plt.subplots(1, 2, figsize=(11.2, 5.0), constrained_layout=True)
for axis, shock_fraction in zip(axes, [0.25, 0.85]):
    part = seed_results.loc[seed_results["Shock fraction"] == shock_fraction]
    groups = [part.loc[part["Structure"] == name, "Total unpaid"].to_numpy() for name in structures]
    box = axis.boxplot(groups, tick_labels=list(structures), patch_artist=True, showfliers=False)
    for patch, color in zip(box["boxes"], [PALETTE["cyan"], PALETTE["amber"], PALETTE["green"], PALETTE["rose"]]):
        patch.set_facecolor(color)
        patch.set_alpha(0.8)
    axis.set_title(f"Shock fraction {shock_fraction:g}")
    axis.set_ylabel("Total unpaid liabilities")
    axis.tick_params(axis="x", rotation=18)
    finish_axis(axis)
save_figure(figure, "seed_unpaid_distributions")

mean_plot = seed_summary[["Shock fraction", "Structure", "mean"]].rename(columns={"mean": "Mean total unpaid"})
mean_plot.to_csv(FIGURE_DATA / "small_large_mean_losses.csv", index=False)
figure, axis = plt.subplots(figsize=(9.6, 5.2), constrained_layout=True)
x = np.arange(len(structures))
for offset, shock_fraction, color in [(-0.19, 0.25, PALETTE["cyan"]), (0.19, 0.85, PALETTE["rose"])]:
    part = mean_plot.loc[mean_plot["Shock fraction"] == shock_fraction].set_index("Structure").loc[list(structures)]
    axis.bar(x + offset, part["Mean total unpaid"], width=0.38, color=color, label=f"Shock {shock_fraction:g}")
axis.set_xticks(x, list(structures), rotation=18)
axis.set_ylabel("Mean total unpaid liabilities")
axis.set_title("Network Rankings Depend on the Declared Shock Size")
axis.legend()
finish_axis(axis)
save_figure(figure, "small_large_mean_losses")

,Setting,Value
0,Generated runs,200
1,First seed,"290,200"
2,Last seed,"290,399"
3,Institutions per run,12
4,Shock fractions,0.25 and 0.85
5,Rows per structure,400


,Shock fraction,Structure,mean,std,median,min,max
0,0.25,Complete,0.235897,1.4191,0,0,12.625008
1,0.25,Core-periphery,4.464246,8.291147,2.84217e-14,0,47.829114
2,0.25,Ring,2.192639,6.177604,7.10543e-15,0,42.623623
3,0.25,Two-community,0.349516,1.759705,0,0,15.055888
4,0.85,Complete,18.989933,37.636905,1.42109e-14,0,197.777861
5,0.85,Core-periphery,32.144439,44.820174,18.17165,0,237.212135
6,0.85,Ring,27.702364,41.958697,8.028831,0,221.867448
7,0.85,Two-community,20.007676,38.518665,1.42109e-14,0,204.782375


,Shock fraction,Mean-loss rank,Structure,Mean total unpaid
0,0.25,1,Complete,0.235897
1,0.25,2,Two-community,0.349516
2,0.25,3,Ring,2.192639
3,0.25,4,Core-periphery,4.464246
4,0.85,1,Complete,18.989933
5,0.85,2,Two-community,20.007676
6,0.85,3,Ring,27.702364
7,0.85,4,Core-periphery,32.144439


<Figure size 1120x500 with 2 Axes>

<Figure size 960x520 with 1 Axes>